<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h1>Notebook Modelisation - ALS (Spark MLlib)</h1>
<strong>Alternating Least Squares matrix factorization</strong>
<p>Entraine un modele ALS sur les splits temporels preparés dans l'EDA<br>
selectionne les hyperparametres sur le jeu `validation` et evalue sur le jeu`test`.</p>
</div>

In [ ]:
# OPTION A (rapide) - Mettre a jour le repo existant dans /content/sparkle-movie

# A utiliser seulement si le repo local Colab est propre (pas de modifications locales).
# Decommenter et executer si necessaire.

# %cd /content/sparkle-movie
# !git branch --show-current
# !git checkout large
# !git pull origin large
# !git log --oneline -n 3

In [ ]:
# OPTION B (propre) - Repartir d'un clone neuf
# A privilegier si git pull echoue (conflits/modifs locales) ou en cas de doute.
# Decommenter et executer si necessaire.

# %cd /content
# !rm -rf /content/sparkle-movie
# !git clone -b large https://github.com/bruno-coulet/sparkle-movie.git /content/sparkle-movie
# %cd /content/sparkle-movie
# !git log --oneline -n 3

In [ ]:
import os
import sys
import importlib
from pathlib import Path

# Bootstrap Colab robuste: clone runtime prioritaire -> Drive -> fallback clone runtime
in_colab = 'google.colab' in str(get_ipython())

if in_colab:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        # Le montage sera re-tenté plus tard si USE_DRIVE_EXPORTS=True.
        pass

candidate_roots = [
    Path('/content/sparkle-movie'),
    Path('/content/drive/MyDrive/sparkle-movie'),
    Path('/content/drive/MyDrive/sparkle-movie/sparkle-movie'),
]

project_path = next(
    (p for p in candidate_roots if (p / 'src').exists()),
    None,
)

if project_path is None and in_colab:
    project_path = Path('/content/sparkle-movie')
    if not project_path.exists():
        !git clone https://github.com/bruno-coulet/sparkle-movie.git /content/sparkle-movie

if project_path is not None:
    os.chdir(project_path.as_posix())

    # Evite d'importer src.utils depuis une ancienne copie sur Drive.
    sys.path = [p for p in sys.path if '/content/drive/MyDrive/sparkle-movie' not in p]
    if project_path.as_posix() not in sys.path:
        sys.path.insert(0, project_path.as_posix())

    os.environ['SPARKLE_MOVIE_ROOT'] = project_path.as_posix()

import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F

import src.utils as utils
utils = importlib.reload(utils)

print(f'Dossier courant: {Path.cwd()}')
print(f'SPARKLE_MOVIE_ROOT={os.environ.get("SPARKLE_MOVIE_ROOT", "")}')
print(f'utils chargé depuis: {utils.__file__}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier courant: /content/drive/MyDrive/sparkle-movie
SPARKLE_MOVIE_ROOT=/content/drive/MyDrive/sparkle-movie
utils chargé depuis: /content/drive/MyDrive/sparkle-movie/src/utils.py


In [ ]:
# Sélection de la source : "processed_small" ou "processed_big"
DATA_SOURCE = "processed_big"

# Si True: charge les artefacts exportes par eda.ipynb depuis Google Drive
USE_DRIVE_EXPORTS = True
DRIVE_PROCESSED_BASE = "/content/drive/MyDrive/sparkle_movie_processed_data"

# Réglages Spark (utilisés par utils.create_spark_session)
import os
os.environ["SPARKLE_SPARK_MASTER"] = "local[*]"
os.environ["SPARKLE_SPARK_DRIVER_MEMORY"] = "8g"
os.environ["SPARKLE_SPARK_SHUFFLE_PARTITIONS"] = "32"

# seed pour la generation des nombres aleatoires
# reproduis en général le même point de départ de l'ALS
# donc des résultats comparables d’un run à l’autre.
RANDOM_SEED = 42

# Nombre de résultats à afficher dans les classements (top-N)
TOP_N = 5

In [ ]:
if DATA_SOURCE not in {"processed_small", "processed_big"}:
    raise ValueError(
        f"DATA_SOURCE invalide pour ce notebook: {DATA_SOURCE}. Utiliser processed_small ou processed_big."
    )

if DATA_SOURCE == "processed_big":
    DATA_SIZE = "big"
elif DATA_SOURCE == "processed_small":
    DATA_SIZE = "small"
else:
    raise ValueError(f"DATA_SOURCE invalide pour le calcul de DATA_SIZE: {DATA_SOURCE}")

if USE_DRIVE_EXPORTS:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            "USE_DRIVE_EXPORTS=True requiert Google Colab + Google Drive monte."
        ) from exc

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SMALL_BEST_PARAMS = (40, 0.1, 10)  # (rank, regParam, maxIter)

# RMSE test de reference observee sur le petit dataset.
# Mets la valeur mesuree dans ton run small pour une comparaison fiable.
SMALL_REFERENCE_RMSE_TEST = 0.902036

# Tolerance acceptee entre le run courant et la reference small.
RMSE_TOLERANCE = 0.03

# Si True, on tente d'abord SMALL_BEST_PARAMS sur big et on lance la grille uniquement si necessaire.
USE_SMALL_BEST_PARAMS_FIRST = True

RELEVANCE_THRESHOLD = 3.8

# Grille de secours si les metriques ne sont pas conformes
GRID_RANK = [20, 40, 80]
GRID_REG_PARAM = [0.01, 0.05, 0.1, 0.2]
GRID_MAX_ITER = [10, 15, 20]

In [ ]:
spark = utils.create_spark_session()
project_root = utils.get_project_root()

spark.sparkContext.setCheckpointDir("/tmp/checkpoints")

# Recalcule la taille depuis DATA_SOURCE pour éviter les variables résiduelles en mémoire.
if DATA_SOURCE == "processed_big":
    data_size = "big"
elif DATA_SOURCE == "processed_small":
    data_size = "small"
else:
    raise ValueError(f"DATA_SOURCE invalide: {DATA_SOURCE}")

if USE_DRIVE_EXPORTS:
    processed_root = Path(DRIVE_PROCESSED_BASE) / data_size
    path_ratings = processed_root / "ratings_clean.parquet"
    path_movies = processed_root / "movies_clean.parquet"
else:
    # Mode local repository (fallback)
    dataset_format, path_ratings, path_movies = utils.resolve_data_source_paths(
        DATA_SOURCE, project_root=project_root
    )
    if dataset_format != "parquet":
        raise ValueError(
            f"Ce notebook ALS attend une source processed_* en parquet, recu: {DATA_SOURCE} ({dataset_format})"
        )
    processed_root = project_root / "data" / "processed" / data_size

split_root = processed_root / "splits_temporal"


In [ ]:

# Verification de l'existence des artefacts necessaires
required_paths = [
    path_ratings,
    path_movies,
    split_root / "train",
    split_root / "validation",
    split_root / "test",
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Artefacts manquants: {missing}")

# Création des DataFrame à partir des fichiers Parquet
df_movies_clean = spark.read.parquet(path_movies.as_posix())
df_train = spark.read.parquet((split_root / "train").as_posix())
df_val = spark.read.parquet((split_root / "validation").as_posix())
df_test = spark.read.parquet((split_root / "test").as_posix())

'''
Casting (typage) des colonnes pour ALS
userId et movieId en int
rating en float
timestamp en long
'''
for name in ["df_train", "df_val", "df_test"]:
    df = globals()[name]
    globals()[name] = df.select(
        F.col("userId").cast("int").alias("userId"),
        F.col("movieId").cast("int").alias("movieId"),
        F.col("rating").cast("float").alias("rating"),
        F.col("timestamp").cast("long").alias("timestamp"),
    )

print(f"Artefacts charges avec succes depuis DATA_SOURCE={DATA_SOURCE}.")
print(f"Taille detectee: {data_size}")
print(f"Source artefacts: {processed_root}")

Artefacts charges avec succes depuis DATA_SOURCE=processed_big.
Taille detectee: big
Source artefacts: /content/drive/MyDrive/sparkle_movie_processed_data/big


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Chargement des artefacts</h3>
<i>artefact : fichier produit par une étape du pipeline et réutilisé plus tard.</i><br>
Si le message affiche <b>"Artefacts charges avec succes"</b>, la base experimentale est prete.

Verification implicite:
- Les fichiers nettoyes et les splits temporels existent bien.
- Les colonnes userId, movieId, rating, timestamp ont ete castees au bon type pour ALS.

En cas d'erreur, il faut relancer le notebook EDA pour regenerer les artefacts.
</div>

In [ ]:
print("--- Tailles des splits ---")
print(f"Train: {df_train.count()}")
print(f"Validation: {df_val.count()}")
print(f"Test: {df_test.count()}")

print("\n--- Cardinalites train ---")
print(f"Users train: {df_train.select('userId').distinct().count()}")
print(f"Items train: {df_train.select('movieId').distinct().count()}")

--- Tailles des splits ---
Train: 27051983
Validation: 3397022
Test: 3383157

--- Cardinalites train ---
Users train: 280324
Items train: 49322


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Taille des splits</h3>
Cette sortie confirme la proportion train/validation/test et la cardinalite users/items en train.

Comment interpreter:
- Train doit etre majoritaire pour apprendre correctement.
- Validation sert au choix des hyperparametres.
- Test reste strictement reserve a l'evaluation finale.

Si les cardinalites sont tres faibles, il faut reduire la complexite du modele (rank, iterations).
</div>

In [ ]:
# 1. Configuration du Checkpoint (Indispensable pour la mémoire Spark)
checkpoint_dir = "checkpoints"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

# 2. Préparation des données
df_train_val = df_train.unionByName(df_val)

# 3. Paramètres ciblés (Rank augmenté pour booster Precision/Recall/Coverage)
# Le rank 100 permet de mieux capturer la diversité du catalogue Movielens Large
import os

# 1. Configuration du Checkpoint (Indispensable pour la mémoire Spark)
checkpoint_dir = "checkpoints"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

# 2. Préparation des données
df_train_val = df_train.unionByName(df_val)

# 3. Paramètres ciblés (Rank augmenté pour booster Precision/Recall/Coverage)
# Le rank 100 permet de mieux capturer la diversité du catalogue Movielens Large
tuned_rank = 100
tuned_reg = 0.1
tuned_iter = 15

print(f"Lancement de l'entraînement (Rank={tuned_rank}, Reg={tuned_reg})...")

# 4. Définition et Entraînement du modèle
als_final = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=tuned_rank,
    regParam=tuned_reg,
    maxIter=tuned_iter,
    coldStartStrategy="drop",
    nonnegative=True,
    checkpointInterval=10, # Nettoie le lignage Spark pour éviter le crash
    seed=RANDOM_SEED
)

# On entraîne sur l'union Train + Val pour avoir un maximum de données
model = als_final.fit(df_train_val)

print("✅ Modèle entraîné avec succès sur le grand dataset.")

# 5. Évaluation rapide du RMSE sur le Test pour vérification
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
predictions_test = model.transform(df_test)
rmse_test = evaluator_rmse.evaluate(predictions_test)

print(f"RMSE final sur le jeu de Test : {rmse_test:.4f}")


Lancement de l'entraînement (Rank=100, Reg=0.1)...
✅ Modèle entraîné avec succès sur le grand dataset.
RMSE final sur le jeu de Test : 0.8379
Lancement de l'entraînement (Rank=100, Reg=0.1)...
✅ Modèle entraîné avec succès sur le grand dataset.
RMSE final sur le jeu de Test : 0.8379


In [ ]:
# Sauvegarde du modele ALS final (artefact reutilisable)
from pathlib import Path
from datetime import datetime

# Verification defensive: le modele final doit avoir ete entraine dans les cellules precedentes
if "model_final" not in globals():
    raise NameError("model_final est introuvable. Execute d'abord les cellules d'entrainement ALS.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if USE_DRIVE_EXPORTS:
    base_output_dir = Path(DRIVE_PROCESSED_BASE) / data_size / "artifacts" / "als_models"
else:
    # Le notebook est dans notebooks/, donc ../artifacts pointe vers la racine du projet
    base_output_dir = Path("../artifacts/als_models").resolve()

model_output_dir = base_output_dir / f"als_model_{timestamp}"

model_output_dir.mkdir(parents=True, exist_ok=True)

# Ecriture du modele Spark MLlib (sans overwrite pour creer un nouveau dossier a chaque fois)
model_final.write().save(str(model_output_dir))

print(f"Modele ALS sauvegarde dans: {model_output_dir}")

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Selection des hyperparametres ALS</h3>
Le notebook applique d'abord les best params du petit dataset, puis decide s'il faut lancer une grille de secours.

Regle de decision:
- Si la RMSE test obtenue avec les params small reste dans la tolerance fixee, on conserve ces params.
- Sinon, une grid search est lancee sur validation pour recalibrer le modele.

Interpretation:
- Cette approche reduit le temps de calcul sur le grand dataset.
- Elle conserve une strategie de repli automatique si les performances se degradent.

Bonnes pratiques:
- Renseigner SMALL_REFERENCE_RMSE_TEST avec la vraie valeur du run small.
- Ajuster RMSE_TOLERANCE (ex: 0.01 a 0.03) selon ton niveau d'exigence.
</div>

In [ ]:
'''
Re-entraine sur train + validation avec les meilleurs parmètres
entraîne le modèle final avec un maximum de données non-test
pour améliorer la généralisation.
'''
df_train_val = df_train.unionByName(df_val)

best_rank, best_reg_param, best_max_iter = best_params
als_final = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=best_rank,
    regParam=best_reg_param,
    maxIter=best_max_iter,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=RANDOM_SEED,
)

model_final = als_final.fit(df_train_val)

# Evaluation finale sur le test set
pred_test = model_final.transform(df_test).dropna(subset=["prediction"])
rmse_test = evaluator.evaluate(pred_test)

print(f"RMSE test (modele final): {rmse_test:.4f}")

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - RMSE test</h3>
Ce score est la mesure principale de performance hors echantillon.

Interpretation:
- Plus le RMSE test est bas, plus la prediction des notes est precise.
- Si RMSE du jeu de test est nettement pire que celle du jeu de validation<br>
  il peut y avoir surapprentissage.

Ordres de grandeur utiles:
- Une baisse de RMSE de 0.01 a 0.03 est deja interessante sur MovieLens.
- Une baisse plus forte indique souvent un vrai gain de calibration du modele.

Limite a garder en tete:
- Le RMSE evalue la prediction de note, pas directement la qualite du top-N.
- C'est pour cela qu'on ajoute ensuite Precision@K, Recall@K et Coverage@K.

Ce score servira de reference pour comparer ensuite contenu et KNN.
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Notes réelles vs notes prédites</h3>
Le tableau ci dessous compare les notes reelles du test et les notes predites par ALS pour quelques utilisateurs.

Interpretation des colonnes:
- userId: utilisateur cible.
- movieId / title: film evalue.
- note_relle_test: note observee dans le jeu test.
- note_predite: note estimee par ALS.

Point important:
- Ici, tous les films affiches ont une note reelle dans le test (pas de ligne vide).
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Métriques</h3>
</div>


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<strong>Precision@K (La pertinence)</strong><br>
Sur les $K$ films recommandés, combien sont réellement pertinents pour l'utilisateur ?<br>
Si on affiche un top 5 sur l'application,<br>
une Precision@5 de 0.8 signifie que 4 films sur 5 plaisent à l'utilisateur.
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">

<strong>Recall@K (L'exhaustivité)</strong><br>
Sur tous les films que l'utilisateur aime vraiment, quelle proportion ai-je réussi à capturer dans mon top K ?<br>
Contrairement à la précision, le dénominateur est le nombre total de "coups de cœur" possibles de l'utilisateur
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<strong>Coverage@K (La diversité du catalogue)</strong><br>
La Couverture ne regarde pas l'utilisateur, mais le catalogue.<br>
Sur l'ensemble de la base de films, quel pourcentage est effectivement recommandé au moins une fois dans les tops K des utilisateurs ?<br>
Un modèle peut avoir une excellente Precision@K en recommandant uniquement Titanic et Avatar à tout le monde.<br>
Mais le Coverage sera très faible (proche de 0%).<br>
Permet d'éviter l'effet "bulle" et s'assurer que les films de niche sont aussi proposés
</div>

Precision : $\frac{\text{Nombre de recommandations pertinentes parmi les } K \text{ premières}}{K}$

Recall : $\frac{\text{Nombre de recommandations pertinentes parmi les } K \text{ premières}}{\text{Nombre total d'items pertinents pour cet utilisateur}}$

In [ ]:
# Evaluation Top-K commune (ALS baseline): Precision@K, Recall@K, Coverage@K
# A executer apres l'entrainement de model_final.
K = TOP_N


# 1) Utilisateurs a evaluer: presents a la fois dans train_val et dans test.
# On ne mesure la qualite que la ou une verite terrain existe (interactions test).
df_users_train_val = df_train_val.select("userId").distinct()
df_users_test = df_test.select("userId").distinct()
df_eval_users = df_users_train_val.join(df_users_test, on="userId", how="inner")

# 2) Recommandations ALS Top-K pour ces utilisateurs
df_reco_eval = model_final.recommendForUserSubset(df_eval_users, K)
df_reco_eval_flat = (
    df_reco_eval.withColumn(
        "rec",
        F.explode("recommendations")
    ).select(
        "userId",
        F.col("rec.movieId").cast("int").alias("movieId"),
        F.col("rec.rating").alias("score")
    )
)

# 3) Verite terrain test: item pertinent si rating >= seuil
df_relevant = (
    df_test
    .filter(F.col("rating") >= RELEVANCE_THRESHOLD)
    .select("userId", "movieId")
    .distinct()
)

# 4) True positives = recommandations qui apparaissent aussi dans les items pertinents
df_hits = df_reco_eval_flat.join(df_relevant, on=["userId", "movieId"], how="inner")
df_hits_per_user = df_hits.groupBy("userId").agg(F.count("*").alias("tp"))

# Nombre de recommandations par utilisateur (robuste meme si < K dans certains cas rares)
df_rec_per_user = df_reco_eval_flat.groupBy("userId").agg(F.count("*").alias("n_rec"))

# Nombre d'items pertinents reels par utilisateur
df_rel_per_user = df_relevant.groupBy("userId").agg(F.count("*").alias("n_rel"))

# 5) Precision@K et Recall@K par utilisateur puis moyenne macro
df_user_metrics = (
    df_rec_per_user
    .join(df_rel_per_user, on="userId", how="left")
    .join(df_hits_per_user, on="userId", how="left")
    .fillna({"n_rel": 0, "tp": 0})
    .withColumn("precision_at_k", F.col("tp") / F.col("n_rec"))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_rel") > 0, F.col("tp") / F.col("n_rel")).otherwise(F.lit(0.0))
    )
)

precision_at_k = df_user_metrics.agg(F.avg("precision_at_k").alias("p")).first()["p"]
recall_at_k = df_user_metrics.agg(F.avg("recall_at_k").alias("r")).first()["r"]

# 6) Coverage@K = proportion d'items du catalogue recommandes au moins une fois
n_items_recommended = df_reco_eval_flat.select("movieId").distinct().count()
n_items_catalog = df_train_val.select("movieId").distinct().count()
coverage_at_k = n_items_recommended / n_items_catalog if n_items_catalog > 0 else 0.0

# 7) Tableau de synthese (ligne ALS baseline)
df_metrics_als = pd.DataFrame([
    {
        "method": "ALS",
        "k": K,
        "relevance_threshold": RELEVANCE_THRESHOLD,
        "rmse_test": float(rmse_test),
        "precision_at_k": float(precision_at_k),
        "recall_at_k": float(recall_at_k),
        "coverage_at_k": float(coverage_at_k),
        "n_eval_users": df_eval_users.count()
    }
])

display(df_metrics_als)

In [ ]:
pd.options.display.float_format = '{:,.6f}'.format
display(df_metrics_als)

In [ ]:
# spark.stop()
# print("Session Spark arretee.")

Exports

In [ ]:
# Export des artefacts d'evaluation (params, metriques, resume de run)
import json
from datetime import datetime, timezone
from pathlib import Path

# Define timestamp once at the beginning of the cell, consistent with model export
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if USE_DRIVE_EXPORTS:
    base_export_dir = Path(DRIVE_PROCESSED_BASE) / data_size / "artifacts" / "als_reports"
else:
    export_dir = Path("../artifacts/als_reports").resolve()

# Create a unique directory for each run's reports
export_dir = base_export_dir / f"run_{timestamp}"
export_dir.mkdir(parents=True, exist_ok=True)

# Structure des best params
best_rank, best_reg_param, best_max_iter = best_params

run_summary = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "data_source": DATA_SOURCE,
    "data_size": data_size,
    "use_drive_exports": bool(USE_DRIVE_EXPORTS),
    "random_seed": int(RANDOM_SEED),
    "small_best_params": {
        "rank": int(SMALL_BEST_PARAMS[0]),
        "regParam": float(SMALL_BEST_PARAMS[1]),
        "maxIter": int(SMALL_BEST_PARAMS[2]),
    },
    "selected_best_params": {
        "rank": int(best_rank),
        "regParam": float(best_reg_param),
        "maxIter": int(best_max_iter),
    },
    "rmse_validation_selected": float(best_rmse),
    "rmse_test_final": float(rmse_test),
    "small_reference_rmse_test": (
        float(SMALL_REFERENCE_RMSE_TEST) if SMALL_REFERENCE_RMSE_TEST is not None else None
    ),
    "rmse_tolerance": float(RMSE_TOLERANCE),
    "grid_search_executed": bool(globals().get("run_grid_search", False)),
    "model_output_dir": str(model_output_dir) if "model_output_dir" in globals() else None,
}

summary_path = export_dir / "run_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

# Export de la grille de tuning (1 ligne si pas de grid search)
grid_path = export_dir / "grid_results.csv"
if "df_grid" in globals():
    df_grid.to_csv(grid_path, index=False)

# Export des metriques finales
metrics_path = export_dir / "metrics_als.csv"
if "df_metrics_als" in globals():
    df_metrics_als.to_csv(metrics_path, index=False)

print(f"Rapports exportes dans: {export_dir}")
print(f"- Resume run: {summary_path}")
if grid_path.exists():
    print(f"- Resultats tuning: {grid_path}")
if metrics_path.exists():
    print(f"- Metriques finales: {metrics_path}")